In [ ]:
#! /usr/bin/env python

import rospy 
import actionlib.msg
import actionlib 
import sys

from geometry_msgs.msg import PoseStamped
from nav_msgs.msg import Odometry
from sensor_msgs.msg import LaserScan

import assignment_2_2024.msg
from assignment2_ros1.msg import RobotInfo

import ipywidgets as widgets
from IPython.display import display
import os
import platform

%matplotlib widget

import matplotlib.pyplot as plt
import tf

from tf.transformations import quaternion_matrix 
import numpy as np
from matplotlib.animation import FuncAnimation

## Robot class
The Robot class contains all the actions the robot can perform, including setting and sending a goal, canceling a goal, and receiving status feedback

In [ ]:
class Robot:
    def __init__(self):
        self.status_pub = rospy.Publisher("/robot_status", RobotInfo, queue_size=10)
        self.subscriber = rospy.Subscriber("/odom", Odometry, self.publish_robot_info)
        self.client = actionlib.SimpleActionClient('reaching_goal', assignment_2_2024.msg.PlanningAction)
        self.client.wait_for_server()
        
        self.target_x = 10
        self.target_y = 10
    
        self.latest_feedback = None
        
        self.x_reached, self.y_reached = [], []
        self.x_not_reached, self.y_not_reached = [], []
        
        
    def feedback_callback(self, feedback):
        """
        Function to handle feedback from the action server.

        :param feedback: The feedback message received from the action server.
        :type feedback: PlanningFeedback
        """
        self.latest_feedback = feedback
        
    # this function send the coordinates set
    def sending_goal(self, x, y):
        """
        :param x: widget providing the x-coordinate.
        : type ipywidgets.FloatSlider
        
        :param y: widget providing the y-coordinate.
        : type ipywidgets.FloatSlider.
        """
        
        goal = assignment_2_2024.msg.PlanningGoal()   
        goal.target_pose = PoseStamped()
        goal.target_pose.pose.position.x = x.value
        goal.target_pose.pose.position.y = y.value
        
        
        self.x_not_reached.append(x.value)
        self.y_not_reached.append(y.value)
        
        with self.output:
            self.client.send_goal(goal, feedback_cb = self.feedback_callback)
            rospy.loginfo(f"New goal set and sent ({x.value}, {y.value})")
    
    def publish_robot_info(self, msg):
        """
        Function that publishes the robot position and velocity as a custom message (x, y, vel_x, vel_z) by relying on the values published on the topic /odom

        :param msg: The message to be published
        :type msg: geometry_msgs.msg.TwistStamped

        """
        robot_info = RobotInfo()
        robot_info.x = msg.pose.pose.position.x
        robot_info.y = msg.pose.pose.position.y
        robot_info.vel_x = msg.twist.twist.linear.x
        robot_info.vel_z = msg.twist.twist.angular.z

        self.status_pub.publish(robot_info)
        
    def cancel_goal(self):
        """
        Function that cancel the goal if it is not already reached and notifies the user
        """
        with self.output:
            if self.client.get_state() not in [actionlib.GoalStatus.SUCCEEDED, actionlib.GoalStatus.ABORTED, actionlib.GoalStatus.PREEMPTED]:
                rospy.loginfo("Cancelling goal")
                self.goal_sent = False
                self.client.cancel_goal()
            else:
                rospy.loginfo("Goal already reached")
        
    
    def feedback(self):
        """
        Function that returns a feedback to the users
        """
        with self.output:
            rospy.loginfo("Getting feedback...")
            if self.latest_feedback is None:
                rospy.loginfo("Feedback still not received")
            else:
                rospy.loginfo("Feedback received: %s", self.latest_feedback)

## RobotGUI class
The RobotGUI class manages the graphical user interface for user interaction, giving the possibility to interact with the robot using buttons linked to functions implemented in the Robot class. It provides the following functionalities:

- Setting the robot’s next target using slider_x, slider_y, and the "Set Coordinates" button.
- Canceling the current goal via the "Cancel Goal" button.
- Receiving feedback using the "Feedback" button.
- Clearing the console with the "Clear Console" button to remove displayed content.


## Button class
The Button class improves the readability of the RobotGUI class. The click function allows a specific command to be connected to a button, triggering the assigned function when clicked.

In [ ]:
class RobotGUI:
    def __init__(self, robot):
        """
        Initializes the GUI
        :param robot
        :type Robot
        """
        self.robot = robot
        self.robot.output = widgets.Output()
        
        self.cancel_goal_button = Button("Cancel goal", robot.cancel_goal)
        self.feedback_button = Button("Feedback", robot.feedback)
        self.clear_button = Button("Clear console", self.clear_console)
        
        self.slider_x = widgets.FloatSlider(min=-10, max=10)
        self.slider_y = widgets.FloatSlider(min=-10, max=10)
        float_x = widgets.FloatText(layout=widgets.Layout(width='60px'))
        float_y = widgets.FloatText(layout=widgets.Layout(width='60px'))
        
        x = self.link_float_slider(float_x, self.slider_x)
        y = self.link_float_slider(float_y, self.slider_y)
        
        self.coordinate_button = Button("Set coordinates", lambda: robot.sending_goal(self.slider_x, self.slider_y))

        left_column = widgets.VBox([
            self.cancel_goal_button.object,
            self.feedback_button.object,
            x, y,
            self.coordinate_button.object,
            self.clear_button.object
        ])

        layout = widgets.HBox([
            left_column,
            self.robot.output
        ])
        
        display(layout)
        
    def link_float_slider(self, float_text, slider):
        """
        Link a FloatSlider to a FloatText to easy set of the target
        """
        
        widgets.jslink((float_text, 'value'), (slider, 'value'))
        return widgets.HBox([slider, float_text])
     
    def clear_console(self):
        """
        Clears the robot's output widget display area.
        """
        self.robot.output.clear_output()

class Button:
    def __init__(self, label, command):
        """
        Creates a new button widget and binds a click handler.
        """
        self.label = label
        self.object = widgets.Button(description=self.label)
        self.command = command
        
        self.object.on_click(self.click)
        
    def click(self, _):
        """
        Executes the bound command when the button is clicked.
        """
        self.command()
        


## User interface
This cell initializes an instance of the Robot class along with its corresponding RobotGUI interface and runs them.

In [ ]:
if __name__ == '__main__':
    try:
        rospy.init_node("action_client_node")
        rate = rospy.Rate(10)
        
        robot = Robot()
        robotGui = RobotGUI(robot)
        
    except rospy.ROSInterruptException:
        print("Action client interrupted", file=sys.stderr)        

## Class Visualiser
This class visualizes the robot's position over time by plotting its trajectory using Odometry data from ROS. The "%matplotlib widget" command is essential for enabling interactive plotting.

In [ ]:
class Visualiser:
    def __init__(self):
        self.fig, self.ax = plt.subplots()
        self.ln, = plt.plot([], [], 'k')
        self.x_data, self.y_data = [], []

    def plot_init(self):
        
        self.ax.set_xlim(-10, 10)
        self.ax.set_ylim(-10, 10)
        return self.ln,
        
    def odom_callback(self, msg):
        """
        Callback function for the topic '/odom', used by the subscriber robot_position_sub in the following cell
        :param msg: Odometry message containing the robot's current position
        :type nav_msgs.msg.Odometry
        """
        
        self.y_data.append(msg.pose.pose.position.y)
        self.x_data.append(msg.pose.pose.position.x)

    def update_plot(self, frame):
        """
        Updates the plot with the new position of the robot
        """
        
        self.ln.set_data(self.x_data, self.y_data)
        return self.ln,


## Position of the robot in the environment
This cell uses FuncAnimation to continuously update the robot's position plot. Before running this cell, make sure that the User Interface main cell has already been executed, as the node is not initialized here and the rate is unspecified.

In [ ]:
if __name__ == '__main__':
    try:
        vis = Visualiser()
        
        robot_position_sub = rospy.Subscriber('/odom', Odometry, vis.odom_callback)
        ani = FuncAnimation(vis.fig, vis.update_plot, init_func=vis.plot_init) 
        plt.show()

        rate.sleep()          

    except rospy.ROSInterruptException:
        print("Action client interrupted", file=sys.stderr)

## ClosestObstacle class
The ClosestObstacle class displays and updates a FloatText widget with the value of the closest detected obstacle. When the subscriber in the next cell calls the laser_callback function, the closest obstacle distance is computed and updated accordingly.

In [ ]:
class ClosestObstacle:
    def __init__(self):
        """
        Initializes the widget with default values and displays it.
        """
        self.minimum = 0 
        self.widget = widgets.FloatText(value = self.minimum, disabled = True)
        
        self.label = widgets.Label("Minimum distance:")
        hbox = widgets.HBox([self.label, self.widget])
        display(hbox) 
        
    def laser_callback(self, msg):
        """
        Callback for the LaserScan topic that updates the widget with the closest range.
        """
        
        self.minimum = min(msg.ranges)
        if (self.minimum < msg.range_min): self.minimum = msg.range_min
        self.widget.value = self.minimum
    

## Distance from the closest obstacle
The main function initializes an instance of the ClosestObstacle class and subscribes to the laser_callback function.

In [ ]:
if __name__ == '__main__':
    try:
        closer_obstacle = ClosestObstacle()
        scan_subscriber = rospy.Subscriber("/scan", LaserScan, closer_obstacle.laser_callback)
    except rospy.ROSInterruptException:
        print("Action client interrupted", file=sys.stderr)

## TargetVisualizer class
This class visualizes the number of targets that have been reached and not reached using both a Cartesian plot and a pie chart.

In [ ]:
class TargetVisualizer:
    def __init__(self, robot):
        self.robot = robot
        self.fig, (self.ax, self.ax_hist) = plt.subplots(1, 2, figsize=(10, 5))
        
        # Cartesian graph
        self.reached_ln, = self.ax.plot([], [], 'bo', label = "Reached")
        self.not_reached_ln, = self.ax.plot([], [], 'ro', label = "Not reached")
        self.x_reached, self.y_reached = [], []
        self.x_not_reached = self.robot.x_not_reached
        self.y_not_reached = self.robot.y_not_reached
        
        self.last_x = None
        self.last_y = None
        

    def plot_init(self):
        # Cartesian graph
        self.ax.set_xlim(-10, 10)
        self.ax.set_ylim(-10, 10)
        self.ax.set_xlabel("x axis")
        self.ax.set_ylabel("y axis")
        self.ax.legend()

        
        # Hist graph
        self.hist_label = ["Reached", "Not reached"]
        
        return self.reached_ln, self.not_reached_ln
    
       
    def new_point(self, msg):
        """
        Processes new target points and updates reached/not reached lists.
        """
        self.last_x = msg.pose.pose.position.x
        self.last_y = msg.pose.pose.position.y
        current_x = float(rospy.get_param("/des_pos_x"))
        current_y = float(rospy.get_param("/des_pos_y"))
            
        if (self.robot.client.get_state() is actionlib.GoalStatus.SUCCEEDED and 
           current_x not in self.x_reached and current_y not in self.y_reached):
            try:
                self.x_not_reached.pop()
                self.y_not_reached.pop()
            except IndexError:
                pass
            self.x_reached.append(current_x)
            self.y_reached.append(current_y)

        
    
    def update_plot(self, frame):
        """
        Updates the visualization with the latest target data.
        """
        # Cartesian graph
        self.reached_ln.set_data(self.x_reached, self.y_reached)
        self.not_reached_ln.set_data(self.x_not_reached, self.y_not_reached)
        
        
        # Hist graph
        self.ax_hist.clear()
        self.bar_container = self.ax_hist.bar(self.hist_label, [len(self.x_reached), len(self.x_not_reached)])
        
        return self.reached_ln, self.not_reached_ln, self.bar_container

## Plot for the number of reached/not-reached targets
The main function creates an object for the TargetVisualizer class of the previews cell.

In [ ]:
if __name__ == '__main__':
    try:

        target_vis = TargetVisualizer(robot)
        target_sub = rospy.Subscriber('/odom', Odometry, target_vis.new_point)
        cart = FuncAnimation(target_vis.fig, target_vis.update_plot, init_func = target_vis.plot_init)
        plt.show()

        rate.sleep()          

    except rospy.ROSInterruptException:
        print("Action client interrupted", file=sys.stderr)